In [1]:

%matplotlib qt
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.stats import mode

In [6]:
import os
from pathlib import Path

print("Текущая папка:", os.getcwd())

for p in Path(os.getcwd()).rglob("lr_annotations_31_05_2026.csv"):
    print("Найден файл:", p.resolve())

Текущая папка: /Users/user/Downloads
Найден файл: /Users/user/Downloads/lr_annotations_31_05_2026.csv


In [7]:
data_path = mne.datasets.sample.data_path()
raw = mne.io.read_raw_fif('/Users/user/Downloads/3_3 ритм + поздние ответы_raw (2).fif', preload=True)

Opening raw data file /Users/user/Downloads/3_3 ритм + поздние ответы_raw (2).fif...
Isotrak not found
    Range : 0 ... 164899 =      0.000 ...    41.225 secs
Ready.
Reading 0 ... 164899  =      0.000 ...    41.225 secs...


/var/folders/sw/007c5hsj5zggxfp7nts4rq2w0000gn/T/ipykernel_9643/601499223.py:2: RuntimeWarning: This filename (/Users/user/Downloads/3_3 ритм + поздние ответы_raw (2).fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif('/Users/user/Downloads/3_3 ритм + поздние ответы_raw (2).fif', preload=True)


In [8]:
raw.plot(duration=10, n_channels=16)

qt.core.qobject.connect: QObject::connect(QStyleHints, QStyleHints): unique connections require a pointer to member function of a QObject subclass


Using pyopengl with version 3.1.9


<mne_qt_browser._pg_figure.MNEQtBrowser(0x7fc6ed82d500) at 0x17ee73e80>

Channels marked as bad:
none


In [4]:
# Сохраняем разметку в файл прямо рядом с вашим кодом
raw.annotations.save("lr_annotations_31_05_2026.csv", overwrite=True)
print(raw.annotations)

Overwriting existing file.
<Annotations | 1322 segments: ER (23), LR (30), Stimulus (22), ...>


In [9]:
events, event_id = mne.events_from_annotations(raw)
print("Все найденные метки:", event_id)

tmin = 0.0
tmax = 2.0 

epochs = mne.Epochs(
    raw, 
    events=events, 
    event_id=event_id['Stimulus'], # Режем ТОЛЬКО по этой метке
    tmin=tmin, 
    tmax=tmax, 
    preload=True,
    baseline=None # Отключаем коррекцию, если просто смотрим сигнал
)

Used Annotations descriptions: [np.str_('Stimulus')]
Все найденные метки: {np.str_('Stimulus'): 1}
Not setting metadata
9 matching events found


No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 9 events and 8001 original time points ...
0 bad epochs dropped


In [10]:
# Получаем события и их тайминги из аннотаций записи
events, event_id = mne.events_from_annotations(raw)
sfreq = raw.info['sfreq']
annotations = raw.annotations

stim_id = event_id['Stimulus']
er_id = event_id['ER']
lr_id = event_id['LR']

# Находим все временные точки (в сэмплах), где установлен 'Stimulus'
stim_times_samples = events[events[:, 2] == stim_id][:, 0]
stim_onsets_sec = stim_times_samples / sfreq

if len(stim_onsets_sec) < 2:
    raise ValueError("Недостаточно меток 'Stimulus' для расчета шага стимуляции!")

# Автоматически считаем длительность эпохи по расстоянию между стимулами
epoch_duration = stim_onsets_sec[1] - stim_onsets_sec[0]
print(f"Автоматически определенная длительность эпохи: {epoch_duration:.3f} сек.")

Used Annotations descriptions: [np.str_('Stimulus')]


KeyError: 'ER'

In [ ]:
print("🛠️ Запуск адаптивной авторазметки с защитой от съезжания...")

# 1. Извлекаем данные канала 'Art' для точной подстройки
art_data, times = raw.copy().pick(['Art']).get_data(return_times=True)
art_signal = np.abs(art_data[0])
sfreq = raw.info['sfreq']

# Находим ваш первый ручной маркер
user_stimulus = [a for a in raw.annotations if a['description'] == 'Stimulus']
if not user_stimulus:
    raise ValueError("Пожалуйста, разметьте самый первый 'Stimulus' вручную!")
    
user_stimulus = sorted(user_stimulus, key=lambda x: x['onset'])
first_stim_onset = user_stimulus[0]['onset']
ideal_duration = user_stimulus[0]['duration']

# Параметры сетки
step_sec = 0.033  # Базовый шаг
search_window_sec = 0.02  # Окно поиска физического пика вокруг расчетной точки (20 мс)
search_window_samples = int(search_window_sec * sfreq)

raw_duration = raw.times[-1]
all_stim_onsets = [first_stim_onset]
current_onset = first_stim_onset

# 2. Пошагово генерируем метки с динамической коррекцией по каналу Art
while True:
    # Делаем математический шаг вперед
    expected_onset = current_onset + step_sec
    if expected_onset >= raw_duration:
        break
        
    # Переводим ожидаемое время в индекс сэмпла
    expected_sample = np.searchsorted(times, expected_onset)
    
    # Определяем границы окрестности для поиска реального пика
    start_idx = max(0, expected_sample - search_window_samples)
    end_idx = min(len(art_signal), expected_sample + search_window_samples)
    
    # Ищем самый сильный всплеск на канале Art в этом узком окошке
    local_window = art_signal[start_idx:end_idx]
    
    if len(local_window) > 0 and np.max(local_window) > np.mean(art_signal) * 3:
        # Если реальный пик найден — привязываемся строго к его вершине!
        actual_peak_idx = start_idx + np.argmax(local_window)
        current_onset = times[actual_peak_idx]
    else:
        # Если пика нет (пропуск в записи), берем математическое ожидание
        current_onset = expected_onset
        
    all_stim_onsets.append(current_onset)

print(f"📈 Сгенерировано {len(all_stim_onsets)} адаптивных меток.")

# 3. Записываем скорректированные аннотации в MNE
stim_durations = [ideal_duration] * len(all_stim_onsets)
stim_descriptions = ['Stimulus_Auto'] * len(all_stim_onsets)

auto_stim_annots = mne.Annotations(
    onset=all_stim_onsets,
    duration=stim_durations,
    description=stim_descriptions,
    orig_time=raw.annotations.orig_time
)

clean_user_annotations = mne.Annotations(
    onset=[a['onset'] for a in raw.annotations if a['description'] != 'Stimulus_Auto'],
    duration=[a['duration'] for a in raw.annotations if a['description'] != 'Stimulus_Auto'],
    description=[a['description'] for a in raw.annotations if a['description'] != 'Stimulus_Auto'],
    orig_time=raw.annotations.orig_time
)

raw.set_annotations(clean_user_annotations + auto_stim_annots)
print("✅ Готово! Сетка стимулов успешно адаптирована под дрейф времени прибора.")
raw.plot()

🛠️ Запуск адаптивной авторазметки с защитой от съезжания...
📈 Сгенерировано 1247 адаптивных меток.
✅ Готово! Сетка стимулов успешно адаптирована под дрейф времени прибора.


qt.core.qobject.connect: QObject::connect(QStyleHints, QStyleHints): unique connections require a pointer to member function of a QObject subclass


Using pyopengl with version 3.1.9


<mne_qt_browser._pg_figure.MNEQtBrowser(0x7f8e86af6c10) at 0x1af3164c0>

Channels marked as bad:
[np.str_('Art')]


In [92]:

# 1. Конвертируем текстовые аннотации Stimulus_Auto в события MNE (Events)
events, event_id = mne.events_from_annotations(raw, regexp='Stimulus_Auto')

# 2. Задаем временное окно эпохи (от стимула до стимула)
tmin = 0.0     # Начало эпохи (0 мс — сам момент стимула)
tmax = 0.033    # Конец эпохи (330 мс — аккурат перед следующим стимулом)

# 3. Нарезаем эпохи
# baseline=None означает, что мы пока не вычитаем среднее пре-стимульное значение
# preload=True загружает данные в оперативную память для быстрой работы
epochs = mne.Epochs(
    raw, 
    events=events, 
    event_id=event_id, 
    tmin=tmin, 
    tmax=tmax, 
    baseline=None, 
    preload=True
)

print(f"🎯 Успешно нарезано эпох: {len(epochs)}")
print(f"📐 Форма массива данных эпох (epochs, channels, times): {epochs.get_data().shape}")

Used Annotations descriptions: [np.str_('Stimulus_Auto')]
Not setting metadata
1247 matching events found
No baseline correction applied


0 projection items activated
Using data from preloaded Raw for 1247 events and 133 original time points ...
1 bad epochs dropped
🎯 Успешно нарезано эпох: 1246
📐 Форма массива данных эпох (epochs, channels, times): (1246, 7, 133)


In [ ]:
print("⏳ Расчет устойчивых параметров LR (медиана и мода) для защиты от выбросов...")

epochs_data = epochs_filtered.get_data(picks=['GM R'])[:, 0, :]
times = epochs_filtered.times

peaks_amplitude = []
peaks_latency_ms = []
lr_durations_ms = []

for i, meta in enumerate(valid_meta):
    for lr in meta['LR']:
        stim_onset = meta['onset']
        start_rel = lr['onset'] - stim_onset
        end_rel = start_rel + lr['duration']
        
        lr_durations_ms.append(lr['duration'] * 1000)
        
        idx_start = np.searchsorted(times, max(0, start_rel))
        idx_end = np.searchsorted(times, min(times[-1], end_rel))
        
        lr_segment = epochs_data[i, idx_start:idx_end]
        lr_times = times[idx_start:idx_end]
        
        if len(lr_segment) > 0:
            # Находим абсолютный пик
            abs_idx = np.argmax(np.abs(lr_segment))
            peak_val = lr_segment[abs_idx]
            peak_time = lr_times[abs_idx] * 1000
            
            peaks_amplitude.append(peak_val)
            peaks_latency_ms.append(peak_time)

# ВЫЧИСЛЯЕМ УСТОЙЧИВЫЕ МЕТРИКИ ВМЕСТО СРЕДНЕГО
if peaks_amplitude:
    median_amp = np.median(peaks_amplitude)
    median_duration = np.median(lr_durations_ms)
    median_latency = np.median(peaks_latency_ms)
    
    # Для моды латентности округлим значения до 1 мс, чтобы найти самое частое подножие пика
    rounded_latencies = np.round(peaks_latency_ms, 0)
    mode_latency, _ = mode(rounded_latencies, keepdims=True)
    mode_latency = mode_latency[0]
else:
    median_amp = median_duration = median_latency = mode_latency = 0

print("\n📊 === РОБУСТНЫЙ (УСТОЙЧИВЫЙ) ПАСПОРТ ВОЛНЫ LR ===")
print(f"✅ Проанализировано ручных волн: {len(peaks_amplitude)}")
print(f"🔹 Медианная амплитуда пика: {median_amp * 1e6:.2f} \u03bcV")
print(f"🔹 Медианное время пика: {median_latency:.1f} ms")
print(f"🔹 Модальное (самое частое) время пика: {mode_latency:.1f} ms")
print(f"🔹 Медианная длительность окна: {median_duration:.1f} ms")
print("==================================================\n")

⏳ Расчет устойчивых параметров LR (медиана и мода) для защиты от выбросов...

📊 === РОБУСТНЫЙ (УСТОЙЧИВЫЙ) ПАСПОРТ ВОЛНЫ LR ===
✅ Проанализировано ручных волн: 16
🔹 Медианная амплитуда пика: 47.36 μV
🔹 Медианное время пика: 12.4 ms
🔹 Модальное (самое частое) время пика: 12.0 ms
🔹 Медианная длительность окна: 1.7 ms



In [65]:
filtered_stim_events = []
valid_meta = []  # Метаданные для разметки блоков в Matplotlib

for stim_sample in stim_times_samples:
    stim_onset_sec = stim_sample / sfreq
    epoch_end_sec = stim_onset_sec + epoch_duration
    
    # Ищем аннотации ER и LR, попавшие внутрь этой конкретной эпохи
    er_periods = [a for a in annotations if a['description'] == 'ER' 
                  if not (a['onset'] + a['duration'] <= stim_onset_sec or a['onset'] >= epoch_end_sec)]
    
    lr_periods = [a for a in annotations if a['description'] == 'LR' 
                  if not (a['onset'] + a['duration'] <= stim_onset_sec or a['onset'] >= epoch_end_sec)]
    
    # Сохраняем стимул, только если внутри есть ОБА периода одновременно
    if er_periods and lr_periods:
        filtered_stim_events.append([stim_sample, 0, stim_id])
        valid_meta.append({
            'onset': stim_onset_sec,
            'ER': er_periods,
            'LR': lr_periods
        })

filtered_stim_events = np.array(filtered_stim_events, dtype=np.int64)
n_epochs = len(filtered_stim_events)
print(f" Найдено подходящих эпох (с комбинацией ER и LR): {n_epochs}")

 Найдено подходящих эпох (с комбинацией ER и LR): 10


In [56]:
epochs = mne.Epochs(
    raw, 
    events=filtered_stim_events, 
    event_id=stim_id,
    tmin=0.0, 
    tmax=epoch_duration,          
    picks=['GM R'],  # Оставляем только нужный канал            
    baseline=None,
    preload=True
)

# Защита от ошибок np.int64 для словаря цветов в MNE
labels_to_show = {k: v for k, v in event_id.items() if k in ['ER', 'LR']}
custom_colors = {
    np.int64(er_id): 'green',  
    np.int64(lr_id): 'blue',   
    np.int64(stim_id): 'lightgray'
}

print("🚀 Запуск интерактивного окна MNE...")
print("📌 ВАЖНО: Закройте открывшееся окно графиков, чтобы разблокировать блокнот для следующей ячейки!")

# Открываем встроенный браузер MNE
epochs.plot(
    n_epochs=10,               
    n_channels=1,              
    scalings='auto',      
    events=events,             
    event_id=labels_to_show, 
    event_color=custom_colors,
    picks='all',
    block=True  # Ждем, пока вы закроете окно, прежде чем выполнять код дальше
)

Not setting metadata
10 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 10 events and 133 original time points ...
0 bad epochs dropped
🚀 Запуск интерактивного окна MNE...
📌 ВАЖНО: Закройте открывшееся окно графиков, чтобы разблокировать блокнот для следующей ячейки!


qt.core.qobject.connect: QObject::connect(QStyleHints, QStyleHints): unique connections require a pointer to member function of a QObject subclass


Using pyopengl with version 3.1.9
Dropped 0 epochs: 
The following epochs were marked as bad and are dropped:
[]
Channels marked as bad:
none


<mne_qt_browser._pg_figure.MNEQtBrowser(0x0) at 0x199be5040>

In [ ]:
# 1. Находим индексы эпох, где есть И волна ER, И волна LR
valid_indices = []
for i, meta in enumerate(valid_meta):
    # Проверяем, что списки ER и LR внутри этой эпохи не пусты
    if meta['ER'] and meta['LR']:
        valid_indices.append(i)

print(f"🔍 Всего в записи найдено {len(valid_indices)} эпох, содержащих одновременно ER и LR.")

# 2. Берем первые 10 эпох (или сколько есть, если вдруг меньше 10)
indices_to_plot = valid_indices[:10]

if len(indices_to_plot) == 0:
    print("Не найдено ни одной эпохи, где одновременно размечены и ER, и LR. Проверьте названия меток.")
else:
    # 3. Строим сетку графиков 2x5 или сколько необходимо
    n_plots = len(indices_to_plot)
    ncols = 5
    nrows = int(np.ceil(n_plots / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3 * nrows), sharex=True, sharey=True)
    axes = axes.flatten() if n_plots > 1 else [axes]
    
    # Получаем данные отфильтрованных эпох (для красивого гладкого отображения)
    epochs_data = epochs_filtered.get_data(picks=['GM R'])[:, 0, :]
    times_ms = epochs_filtered.times * 1000
    
    for plot_idx, epoch_idx in enumerate(indices_to_plot):
        ax = axes[plot_idx]
        signal = epochs_data[epoch_idx] * 1e6  # Переводим в мкВ
        
        ax.plot(times_ms, signal, color='black', lw=1.2, label='Signal')
        
        # Подсвечиваем зоны ER (зеленым) и LR (синим) прямо из вашей ручной разметки
        meta = valid_meta[epoch_idx]
        stim_onset = meta['onset']
        
        # Рисуем ER
        for er in meta['ER']:
            start_ms = (er['onset'] - stim_onset) * 1000
            end_ms = start_ms + (er['duration'] * 1000)
            ax.axvspan(start_ms, end_ms, color='green', alpha=0.2, label='ER' if plot_idx==0 else "")
            
        # Рисуем LR
        for lr in meta['LR']:
            start_ms = (lr['onset'] - stim_onset) * 1000
            end_ms = start_ms + (lr['duration'] * 1000)
            ax.axvspan(start_ms, end_ms, color='blue', alpha=0.2, label='LR' if plot_idx==0 else "")
            
        ax.grid(True, linestyle=':', alpha=0.5)
        
    # Удаляем пустые окошки, если эпох оказалось меньше, чем ячеек в сетке
    for j in range(plot_idx + 1, len(axes)):
        fig.delaxes(axes[j])
        
    # Добавляем общую легенду и подписи
    axes[0].legend(fontsize=8, loc='upper right')
    fig.text(0.5, 0.01, 'Time from Stimulus (ms)', ha='center', fontsize=11)
    fig.text(0.01, 0.5, r'Amplitude ($\mu$V)', va='center', rotation='vertical', fontsize=11)
    plt.suptitle("GM R Channel", fontsize=12, weight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

In [ ]:
epochs_data = epochs_filtered.get_data(picks=['GM R'])[:, 0, :]
times = epochs_filtered.times

all_lr_latencies_ms = []

# Собираем латентности пиков из ВСЕХ 1246 эпох
for i, meta in enumerate(valid_meta):
    for lr in meta['LR']:
        stim_onset = meta['onset']
        start_rel = lr['onset'] - stim_onset
        end_rel = start_rel + lr['duration']
        
        # Переводим в индексы сэмплов
        idx_start = np.searchsorted(times, max(0, start_rel))
        idx_end = np.searchsorted(times, min(times[-1], end_rel))
        
        lr_segment = epochs_data[i, idx_start:idx_end]
        lr_times = times[idx_start:idx_end]
        
        if len(lr_segment) > 0:
            # Находим экстремум (пик) внутри вашей ручной зоны
            abs_idx = np.argmax(np.abs(lr_segment))
            peak_time_ms = lr_times[abs_idx] * 1000  # Переводим в мс
            all_lr_latencies_ms.append(peak_time_ms)

print(f"✅ Успешно рассчитана латентность для {len(all_lr_latencies_ms)} волн LR.")

# Строим гистограмму распределения
if len(all_lr_latencies_ms) > 0:
    plt.figure(figsize=(8, 4))
    
    # Рекомендуется использовать bins='auto' или фиксированное количество (например, 20-30),
    # так как данных много (около 1000+ эпох)
    counts, bins, patches = plt.hist(all_lr_latencies_ms, bins=25, color='royalblue', 
                                     edgecolor='black', alpha=0.7, density=False)
    
    # Считаем ключевую статистику для отображения на графике
    median_lat = np.median(all_lr_latencies_ms)
    mean_lat = np.mean(all_lr_latencies_ms)
    std_lat = np.std(all_lr_latencies_ms)
    
    # Рисуем вертикальную линию медианы
    plt.axvline(median_lat, color='red', linestyle='--', lw=2, 
                label=f'Median: {median_lat:.1f} ms')
    
    plt.title("LR latency distribution (GM R)", fontsize=11, weight='bold')
    plt.xlabel("Time, ms", fontsize=10)
    plt.ylabel("Count", fontsize=10)
    
    # Выводим текстовый блок со статистикой в угол графика
    stats_text = f"Total waves: {len(all_lr_latencies_ms)}\nMean: {mean_lat:.1f} ms\nSD: {std_lat:.1f} ms"
    plt.text(0.95, 0.75, stats_text, transform=plt.gca().transAxes, fontsize=9,
             verticalalignment='top', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))
    
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(fontsize=9, loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Данные для гистограммы отсутствуют.")

⏳ Расчет латентностей для каждой ручной волны LR...
✅ Успешно рассчитана латентность для 16 волн LR.
